# Practical 10: Explainable AI: Grad-CAM on Correct vs. Misclassified Leaves
**Course:** CSE471 — Deep Learning for Computer Vision  
**Unit:** Unit VI (Generative Models, VLMs & Explainability) | **CO:** CO6  
**Problem Statement Context:** SIH26131 — Transparent Model Decision-Making & Agronomic Trust  

---

## 1. Grad-CAM Mathematical Formulation (Selvaraju et al., 2017)
Grad-CAM calculates neuron importance weights $lpha_k^c$ for class $c$ using the gradient of the class score $y^c$ with respect to the activation maps $A^k$ of a convolutional layer:

$$lpha_k^c = rac{1}{Z} \sum_{i=1}^u \sum_{j=1}^v rac{\partial y^c}{\partial A_{i, j}^k}$$

The visual localization map $L_{	ext{Grad-CAM}}^c$ is computed as a weighted linear combination followed by a rectifier (ReLU):

$$L_{	ext{Grad-CAM}}^c = 	ext{ReLU}\left( \sum_k lpha_k^c A^k ight)$$

### Key Objectives:
- **Correct Prediction:** Verify the model focuses on actual pathogenic symptoms (e.g. concentric brown rings).
- **Misclassified Prediction:** Detect spurious correlations (e.g., attending to background soil or water drops instead of leaf tissue).

In [ ]:
import sys
from pathlib import Path
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

from src.models_cnn import ResNet18Scratch
from src.vlm_and_gradcam import GradCAM, overlay_gradcam, compute_saliency_map
from src.transforms import IMAGENET_MEAN, IMAGENET_STD

device = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
print(f"[✓] Active Device: {device}")

## 2. Load Model & Prepare Leaf Test Image

In [ ]:
classes = ["Tomato___Early_Blight", "Tomato___Late_Blight", "Tomato___Healthy", "Potato___Early_Blight", "Rice___Leaf_Blast", "Apple___Apple_Scab"]
model = ResNet18Scratch(in_channels=3, num_classes=len(classes)).to(device)
model.eval()

# Select target convolutional layer (last residual block conv2)
target_layer = model.layer4[1].conv2
gradcam = GradCAM(model, target_layer)

sample_path = REPO_ROOT / "data" / "samples" / "classification" / "lab_plantvillage" / "Tomato___Early_Blight" / "sample_000.jpg"
img_pil = Image.open(sample_path).convert("RGB").resize((224, 224))
raw_rgb = np.array(img_pil)

# Normalize
norm_arr = (raw_rgb.astype(np.float32) / 255.0 - np.array(IMAGENET_MEAN)) / np.array(IMAGENET_STD)
tensor = torch.from_numpy(norm_arr).permute(2, 0, 1).unsqueeze(0).float().to(device)

print(f"[✓] Input tensor ready: {tensor.shape}")

## 3. Compare Visual Attributions: Correct Target vs. Wrong Class

In [ ]:
# Case 1: Grad-CAM for True Class (Tomato Early Blight, index 0)
cam_correct = gradcam.generate_heatmap(tensor, target_class=0)
overlay_correct = overlay_gradcam(raw_rgb, cam_correct)

# Case 2: Grad-CAM for Confused Class (Tomato Healthy, index 2)
cam_wrong = gradcam.generate_heatmap(tensor, target_class=2)
overlay_wrong = overlay_gradcam(raw_rgb, cam_wrong)

# Case 3: Vanilla Saliency Map
saliency = compute_saliency_map(model, tensor, target_class=0)

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
axes[0].imshow(raw_rgb)
axes[0].set_title("Input Tomato Leaf")
axes[0].axis("off")

axes[1].imshow(saliency, cmap="hot")
axes[1].set_title("Vanilla Saliency Map")
axes[1].axis("off")

axes[2].imshow(overlay_correct)
axes[2].set_title("Grad-CAM: Early Blight (Target Class)")
axes[2].axis("off")

axes[3].imshow(overlay_wrong)
axes[3].set_title("Grad-CAM: Healthy (False Hypothesis)")
axes[3].axis("off")

plt.tight_layout()
plt.show()

## 4. Practical Takeaways & Auditing AI Models
1. **Model Validation:** When Grad-CAM highlights lesion spots, the model has learned true causal agronomic patterns rather than memorizing background lighting artifacts.
2. **Debugging Failure Modes:** On misclassified field images, Grad-CAM reveals if shadows or soil reflections distracted the feature extractor.